In [1]:
def mv_weights(Theta_hat, mu, target_variance=0.15**2/12):
    """
    Compute Mean-Variance portfolio weights to maximize return given a target variance.
    
    Solves the constrained optimization:
    max w' mu  subject to  w' Sigma w <= target_variance  and  w' 1 = 1
    
    Parameters:
    -----------
    Theta_hat : np.ndarray, shape (p, p)
        Precision matrix (Sigma^{-1})
    mu : np.ndarray, shape (p,)
        Expected returns
    target_variance : float
        Target maximum portfolio variance
    
    Returns:
    --------
    w_star : np.ndarray, shape (p,)
        Portfolio weights
    """
    p = Theta_hat.shape[0]
    ones_p = np.ones(p)
    
    # Compute key quantities
    A = ones_p @ Theta_hat @ ones_p  # 1' Theta 1
    B = ones_p @ Theta_hat @ mu      # 1' Theta mu  
    C = mu @ Theta_hat @ mu          # mu' Theta mu
    D = A * C - B * B                # Determinant
    
    # Check for singularity
    if np.abs(D) < 1e-10 or np.abs(A) < 1e-10:
        print('SINGULARITY: Falling back to equal weights')
        return ones_p / p
        
    # Minimum possible variance (Global Minimum Variance)
    min_variance = 1.0 / A
    
    # Check feasibility
    if target_variance < min_variance:
        print(f'INFEASIBLE: Target variance {target_variance:.6f} is below GMV variance {min_variance:.6f}. Returning GMV portfolio.')
        return (Theta_hat @ ones_p) / A

    # Map target variance to maximum expected return on the efficient frontier
    # The efficient frontier equation is: variance = (A*mu^2 - 2*B*mu + C) / D
    # Solving for the upper half (max return) yields:
    target_return = (B + np.sqrt(D * (A * target_variance - 1))) / A
    
    # Compute Lagrange multipliers using the derived target_return
    lambda1 = (C - B * target_return) / D
    lambda2 = (A * target_return - B) / D
    
    # Compute weights: w = lambda1 * Theta^{-1} 1 + lambda2 * Theta^{-1} mu
    w_star = lambda1 * (Theta_hat @ ones_p) + lambda2 * (Theta_hat @ mu)
    
    return w_star

def msr_weights(Theta_hat, mu):
    """
    Compute Maximum Sharpe Ratio portfolio weights.
    
    The maximum Sharpe ratio portfolio solves:
    max (w' mu) / sqrt(w' Sigma w)
    
    Solution (when mu represents excess returns):
    w ∝ Sigma^{-1} mu = Theta mu
    
    Then normalize so that sum(w) = 1.
    
    Parameters:
    -----------
    Theta_hat : np.ndarray, shape (p, p)
        Precision matrix (Sigma^{-1})
    mu : np.ndarray, shape (p,)
        Expected excess returns
    
    Returns:
    --------
    w_star : np.ndarray, shape (p,)
        Portfolio weights (sum to 1)
    """
    p = Theta_hat.shape[0]
    ones_p = np.ones(p)
    
    # Compute unnormalized weights: w ∝ Theta mu
    w_unnorm = Theta_hat @ mu
    
    # Normalize to sum to 1
    weight_sum = np.sum(w_unnorm)
    
    if np.abs(weight_sum) < 1e-10:
        print('WARNING: Weight sum near zero, returning equal weights')
        return ones_p / p
    
    w_star = w_unnorm / weight_sum
    
    return w_star

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

def train_random_forest(df, train_start, train_end, features=['mom12m', 'mve', 'bm'], clip_tails=True):
    """
    Train Random Forest Regressor on historical data to predict return magnitude.
    """
    train_df = df[(df['datadate'] >= train_start) & (df['datadate'] <= train_end)].copy()
    
    # Drop rows missing features or the continuous target
    train_df = train_df.dropna(subset=features + ['ret_fwd_1'])
    
    # Winsorize (clip) extreme outliers to protect the Regressor's MSE loss function
    if clip_tails:
        lower_cap = train_df['ret_fwd_1'].quantile(0.01)
        upper_cap = train_df['ret_fwd_1'].quantile(0.99)
        train_df['target'] = train_df['ret_fwd_1'].clip(lower_cap, upper_cap)
    else:
        train_df['target'] = train_df['ret_fwd_1']
    
    X_train = train_df[features]
    y_train = train_df['target']
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    # Use Regressor instead of Classifier
    rf_model = RandomForestRegressor(
        n_estimators=200,      # More trees to smooth variance
        max_depth=8,           # Increased from 5 to allow deeper interactions
        min_samples_leaf=15,   # Decreased from 50 to allow smaller, more specific leaf nodes
        max_features='sqrt',   # Forces trees to use different features, increasing variance
        random_state=42, 
        n_jobs=-1
    )
    rf_model.fit(X_train_scaled, y_train)
    
    print(f"  Training samples: {len(train_df)}")
    print(f"  Training R-squared: {rf_model.score(X_train_scaled, y_train):.4f}")
    
    return rf_model, scaler


def select_stocks_rf_quantiles(df, predict_date, rf_model, scaler, 
                                         features=['mom12m', 'mve', 'bm'],
                                         top_q=0.90, bottom_q=0.10):
    """
    Select top and bottom percentiles of stocks based on predicted expected return.
    """
    predict_df = df[df['datadate'] == predict_date].copy()
    predict_df = predict_df.dropna(subset=features)
    
    if len(predict_df) == 0:
        print(f"  ⚠ No stocks with complete data on {predict_date}")
        return []
    
    X_predict = predict_df[features]
    X_predict_scaled = scaler.transform(X_predict)
    
    # Regressor uses .predict() to output a continuous return estimate, not a probability
    predict_df['expected_return'] = rf_model.predict(X_predict_scaled)
    
    # Calculate quantiles on the predicted magnitude
    upper_bound = predict_df['expected_return'].quantile(top_q)
    lower_bound = predict_df['expected_return'].quantile(bottom_q)
    
    # Screen out the dead zone
    selected_df = predict_df[
        (predict_df['expected_return'] >= upper_bound) | 
        (predict_df['expected_return'] <= lower_bound)
    ]
    
    selected_permnos = selected_df['permno'].tolist()
    
    print(f"  Stocks evaluated: {len(predict_df)}")
    print(f"  Upper expected return bound (Top {100-top_q*100:.0f}%): {upper_bound:.4f}")
    print(f"  Lower expected return bound (Bottom {bottom_q*100:.0f}%): {lower_bound:.4f}")
    print(f"  Stocks selected: {len(selected_permnos)}")
    
    return selected_permnos

In [3]:
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso, LassoCV, LinearRegression
import warnings
warnings.filterwarnings('ignore')

def est_ndwcov_factor(Y, factors, ic, lambda_min=True):
    """
    Estimate nodewise covariance with factor models using LASSO.
    
    Parameters:
    -----------
    Y : numpy.ndarray
        n x p matrix of observations
    factors : numpy.ndarray
        n x k matrix of factors
    ic : str
        Information criterion: 'WIC', 'BIC', 'GIC', 'AIC', or 'cv'
    lambda_min : bool
        If True and ic='cv', use lambda.min; otherwise use lambda.1se
        
    Returns:
    --------
    TAU : numpy.ndarray
        p x p precision matrix estimate
    """
    # Initialization
    p = Y.shape[1]
    n = Y.shape[0]
    C = np.zeros((p, p))
    np.fill_diagonal(C, 1)
    tau = []
    ns1 = np.ones((n, 1))
    
    # Fit factor model: Y = factors * beta + u
    # Add intercept to factors
    factors_with_intercept = np.column_stack([np.ones(n), factors])
    
    # Fit linear regression for each column of Y
    factormodel = LinearRegression(fit_intercept=False)
    factormodel.fit(factors_with_intercept, Y)
    
    # Get residuals and beta coefficients (excluding intercept)
    u = Y - factormodel.predict(factors_with_intercept)
    beta = factormodel.coef_[:, 1:]  # p x k matrix (excluding intercept)
    
    # Loop over the assets
    for j in range(p):
        # Create design matrix excluding column j
        X_j = np.delete(u, j, axis=1)
        y_j = u[:, j]
        
        if ic != 'cv':
            # Fit LASSO path
            alphas = np.logspace(-4, 1, 100)  # Create lambda sequence
            df_list = []
            sig_list = []
            bic_list = []
            coef_list = []
            res_list = []
            
            for alpha in alphas:
                model = Lasso(alpha=alpha, fit_intercept=False, max_iter=10000)
                model.fit(X_j, y_j)
                
                # Predictions and residuals
                y_pred = model.predict(X_j)
                res = y_j - y_pred
                
                # Degrees of freedom (number of non-zero coefficients)
                df = np.sum(np.abs(model.coef_) > 1e-8)
                
                # Variance of residuals
                sig = np.sum(res**2) / n
                
                # Compute information criterion
                if ic == 'WIC':
                    bic_val = np.log(sig) + df * np.log(n) / n * np.log(np.log(p))
                elif ic == 'BIC':
                    bic_val = np.log(sig) + df * np.log(n) / n
                elif ic == 'GIC':
                    bic_val = np.log(sig) + df * np.log(p) * np.log(np.log(n)) / n
                elif ic == 'AIC':
                    bic_val = np.log(sig) + 2 * df
                else:
                    raise ValueError(f"Unknown IC: {ic}")
                
                df_list.append(df)
                sig_list.append(sig)
                bic_list.append(bic_val)
                coef_list.append(model.coef_.copy())
                res_list.append(res)
            
            # Select model with minimum IC
            jind = np.argmin(bic_list)
            jpar = coef_list[jind]
            jres = res_list[jind]
            jtau = np.sum(y_j * jres) / n
            
        else:  # Cross-validation
            lasso_cv = LassoCV(cv=5, fit_intercept=False, max_iter=10000, n_alphas=100)
            lasso_cv.fit(X_j, y_j)
            
            if lambda_min:
                # Use alpha that minimizes CV error (lambda.min equivalent)
                jfit = lasso_cv.predict(X_j)
                jpar = lasso_cv.coef_
            else:
                # Use alpha within 1 SE of minimum (lambda.1se equivalent)
                cv_scores = lasso_cv.mse_path_.mean(axis=1)
                cv_std = lasso_cv.mse_path_.std(axis=1)
                min_idx = np.argmin(cv_scores)
                threshold = cv_scores[min_idx] + cv_std[min_idx]
                
                # Find largest alpha with CV score below threshold
                valid_indices = np.where(cv_scores <= threshold)[0]
                se_idx = valid_indices[0] if len(valid_indices) > 0 else min_idx
                
                selected_alpha = lasso_cv.alphas_[se_idx]
                model_1se = Lasso(alpha=selected_alpha, fit_intercept=False, max_iter=10000)
                model_1se.fit(X_j, y_j)
                jfit = model_1se.predict(X_j)
                jpar = model_1se.coef_
            
            jres = y_j - jfit
            jtau = np.sum(y_j * jres) / n
        
        # Fill in C matrix
        # Insert coefficients back (accounting for missing j-th position)
        C_row = np.insert(-jpar / jtau, j, 0)
        C[j, :] = C_row
        tau.append(jtau)
    
    # Set diagonal
    np.fill_diagonal(C, 1 / np.array(tau))
    omega = C.copy()
    omegasym = (C + C.T) / 2
    
    # Compute factor covariance - ensure float64
    covft = (1/n) * (factors.T @ factors) - (1/(n**2)) * (factors.T @ ns1 @ ns1.T @ factors)
    covft = covft.astype(np.float64)
    
    # Ensure beta and omegasym are float64
    beta = beta.astype(np.float64)
    omegasym = omegasym.astype(np.float64)
    

    covft_inv = np.linalg.inv(covft)
    p1 = np.linalg.inv(covft_inv + beta.T @ omegasym @ beta)
    TAU = omega - omega @ beta @ p1 @ beta.T @ omega
    
    return TAU


def gmv_weights(Theta_hat):
    """
    Compute Global Minimum Variance portfolio weights.
    """
    p = Theta_hat.shape[0]
    ones_p = np.ones(p)
    
    numerator = Theta_hat @ ones_p
    denominator = ones_p @ Theta_hat @ ones_p
    
    if np.abs(denominator) < 1e-10:
        return ones_p / p
    
    w_star = numerator / denominator
    return w_star

def load_ff_factors(factors_path='factors_ff_daily.csv'):
    """
    Load Daily Fama-French factors from CSV file.
    """
    factors_df = pd.read_csv(factors_path)
    
    # Clean the date column: some FF files have trailing spaces or are read as floats
    date_strings = factors_df.iloc[:, 0].astype(str).str.strip().str.replace('.0', '', regex=False)
    
    # Parse using the 8-digit daily format YYYYMMDD
    # (If your CSV has dashes like '2024-01-01', change format to '%Y-%m-%d')
    factors_df['date'] = pd.to_datetime(date_strings, format='%Y%m%d')
    
    # Set date as index and keep only factor columns
    factors_df = factors_df.set_index('date')[['Mkt-RF', 'SMB', 'HML']]
    
    # Convert to decimal form (assuming factors are in percentage points)
    factors_df = factors_df / 100
    
    return factors_df

def integrated_backtest(df_daily, df_monthly,
                              factors_path='F-F_Research_Data_Factors_daily.csv',
                              ic='GIC',
                              test_start_date='2020-01-31',
                              test_end_date='2024-04-30',
                              rf_train_years=15,
                              features=['mom12m', 'mve', 'bm'],
                              top_q=0.90,
                              bottom_q=0.10,
                              lookback_window=60, # 60 daily periods ~ 3 months
                              transaction_cost=0.001, # Adjusted for daily trading (e.g. 10 bps)
                              portfolio_type='all',  
                              mv_target_variance=0.15**2/252,
                              verbose=True):
    """
    Integrated daily backtest combining monthly Random Forest stock selection 
    with daily Residual Nodewise regression portfolio optimization.
    """
    df_daily = df_daily.copy()
    df_monthly = df_monthly.copy()
    
    df_daily['datadate'] = pd.to_datetime(df_daily['datadate'])
    df_monthly['datadate'] = pd.to_datetime(df_monthly['datadate'])
    
    # Load Daily Fama-French factors (assuming load_ff_factors is updated for daily)
    factors_df = load_ff_factors(factors_path)
    
    daily_dates = sorted(df_daily['datadate'].unique())
    
    # Parse test dates
    test_start_dt = pd.to_datetime(test_start_date)
    test_end_dt = pd.to_datetime(test_end_date)
    
    try:
        test_start_idx = daily_dates.index(test_start_dt)
        test_end_idx = daily_dates.index(test_end_dt)
    except ValueError as e:
        raise ValueError(f"Date not found in daily DataFrame: {e}")
    
    if test_start_idx < lookback_window:
        raise ValueError(f"Not enough daily data for lookback window.")
    
    if portfolio_type == 'all':
        portfolio_types = ['gmv', 'mv', 'msr']
    else:
        portfolio_types = [portfolio_type]
    
    # Storage for each portfolio type
    results_storage = {ptype: {
        'returns': [], 'dates': [], 'weights_list': [],
        'turnover_list': [], 'gross_returns': [],
        'prev_weights_dict': {}, 'prev_oos_returns_dict': {},
        'prev_gross_return': 0.0
    } for ptype in portfolio_types}
    
    # Cache to store RF selected stocks so we only train once per month
    rf_universe_cache = {}
    
    if verbose:
        print("="*70)
        print("DAILY INTEGRATED BACKTEST: RF SELECTION + RESIDUAL NODEWISE")
        print(f"Test Period: {test_start_date} to {test_end_date}")
        print("="*70)
    
    # Daily loop
    for t in range(test_start_idx, test_end_idx + 1):
        current_date = daily_dates[t]
        
        # --- 1. MONTHLY RF STOCK SELECTION ---
        # Calculate the previous month (T-1) for safe, lookahead-free signal lookup
        prev_month_period = (current_date - pd.DateOffset(months=1)).to_period('M')
        
        if prev_month_period not in rf_universe_cache:
            month_mask = df_monthly['datadate'].dt.to_period('M') == prev_month_period
            if not month_mask.any():
                if verbose:
                    print(f"\n⚠ No monthly data found for {prev_month_period}, skipping month")
                rf_universe_cache[prev_month_period] = []
            else:
                retrain_date = df_monthly[month_mask]['datadate'].max()
                
                if verbose:
                    print(f"\n{'='*70}")
                    print(f"TRAINING RF FOR TRADING MONTH: {current_date.strftime('%B %Y')}")
                    print(f"Using monthly features at: {retrain_date.strftime('%Y-%m-%d')}")
                    print(f"{'='*70}")
                
                train_end = retrain_date - pd.DateOffset(months=1)
                train_start = retrain_date - pd.DateOffset(years=rf_train_years)
                
                rf_model, scaler = train_random_forest(
                    df_monthly, train_start, train_end, features=features
                )
                
                current_permnos = select_stocks_rf_quantiles(
                    df_monthly, retrain_date, rf_model, scaler=scaler,
                    features=features, top_q=top_q, bottom_q=bottom_q
                )
                rf_universe_cache[prev_month_period] = current_permnos
                
        current_permnos = rf_universe_cache[prev_month_period]
        
        if len(current_permnos) == 0:
            if verbose:
                print(f"[{current_date.strftime('%Y-%m-%d')}] ⚠ No stocks in RF universe, skipping")
            continue

        # --- 2. DAILY RESIDUAL NODEWISE REGRESSION ---
        window_start_date = daily_dates[t - lookback_window]
        window_end_date = daily_dates[t - 1]
        
        # Filter daily data to RF selected stocks only
        train_data = df_daily[
            (df_daily['datadate'] >= window_start_date) & 
            (df_daily['datadate'] <= window_end_date) &
            (df_daily['permno'].isin(current_permnos))
        ]
        
        returns_pivot = train_data.pivot(index='datadate', columns='permno', values='ret_fwd_1')
        window_dates = daily_dates[t - lookback_window : t]
        returns_pivot = returns_pivot.reindex(index=window_dates)
        
        # =========================================================
        # CONTEMPORANEOUS DAILY FACTOR ALIGNMENT
        # =========================================================
        try:
            # Reindex to exactly match our daily window_dates
            factors_window = factors_df.reindex(window_dates)
            
            # Forward-fill any slight holiday/calendar mismatches between 
            # your stock universe dates and FF factor dates
            factors_window = factors_window.ffill().bfill()
        except Exception as e:
            raise ValueError(f"Error aligning daily factors: {e}")
            
        if factors_window.isna().any().any():
            if verbose:
                print(f"  ⚠ Missing factor data in window, skipping period")
            continue
        # =========================================================
        
        nan_assets = returns_pivot.columns[returns_pivot.isna().any()]
        filtered_pivot = returns_pivot.drop(columns=nan_assets)
        
        current_assets = filtered_pivot.columns.tolist()
        Y = filtered_pivot.values
        factors = factors_window.values
        n_train, p_current = Y.shape
        
        if verbose:
            print(f"\n[{current_date.strftime('%Y-%m-%d')}] | Assets: {p_current}/{len(current_permnos)} valid")
        
        if n_train < lookback_window or p_current < 2:
            if verbose:
                print(f"  ⚠ Insufficient daily data, using previous weights")
            new_weights_dict = {ptype: results_storage[ptype]['prev_weights_dict'].copy() 
                               for ptype in portfolio_types}
        else:
            try:
                Y_bar = Y.mean(axis=0)
                Y_star = Y - Y_bar
                
                if verbose:
                    print(f"  Running Residual Nodewise...")
                    
                # Calculate Theta_hat using the aligned factors
                Theta_hat = est_ndwcov_factor(Y, factors, ic=ic, lambda_min=True)
                
                new_weights_dict = {}
                
                if 'gmv' in portfolio_types:
                    w_gmv = gmv_weights(Theta_hat)
                    new_weights_dict['gmv'] = {asset: w_gmv[i] for i, asset in enumerate(current_assets)}
                
                if 'mv' in portfolio_types or 'msr' in portfolio_types:
                    mu = Y_bar
                    if 'mv' in portfolio_types:
                        w_mv = mv_weights(Theta_hat, mu, target_variance=mv_target_variance)
                        new_weights_dict['mv'] = {asset: w_mv[i] for i, asset in enumerate(current_assets)}
                    
                    if 'msr' in portfolio_types:
                        w_msr = msr_weights(Theta_hat, mu)
                        new_weights_dict['msr'] = {asset: w_msr[i] for i, asset in enumerate(current_assets)}
                        
            except Exception as e:
                if verbose:
                    print(f"  ✗ Error: {e}")
                new_weights_dict = {ptype: results_storage[ptype]['prev_weights_dict'].copy() 
                                   for ptype in portfolio_types}
        
        # --- 3. OOS RETURNS & TRANSACTION COSTS ---
        oos_data = df_daily[df_daily['datadate'] == current_date]
        oos_returns_series = oos_data.set_index('permno')['ret_fwd_1'].dropna()
        oos_returns_dict = oos_returns_series.to_dict()
        
        for ptype in portfolio_types:
            weights = new_weights_dict[ptype]
            weight_sum = sum(weights.values())
            if weight_sum > 1e-10:
                weights = {k: v/weight_sum for k, v in weights.items()}
            else:
                weights = results_storage[ptype]['prev_weights_dict'].copy()
                
            prev_weights = results_storage[ptype]['prev_weights_dict']
            prev_oos_returns = results_storage[ptype]['prev_oos_returns_dict']
            prev_gross_ret = results_storage[ptype]['prev_gross_return']
            
            common_assets = set(weights.keys()) & set(oos_returns_dict.keys())
            
            if len(common_assets) == 0:
                continue
            
            common_weights = {a: weights[a] for a in common_assets}
            common_weight_sum = sum(common_weights.values())
            if common_weight_sum > 1e-10:
                common_weights = {k: v/common_weight_sum for k, v in common_weights.items()}
            else:
                continue
            
            gross_return = sum(common_weights[a] * oos_returns_dict[a] for a in common_assets)
            if np.isnan(gross_return) or np.isinf(gross_return):
                continue
            
            # Daily Transaction Costs
            if len(prev_weights) > 0:
                adjusted_prev = {}
                for asset, prev_w in prev_weights.items():
                    if asset in prev_oos_returns:
                        prev_r = prev_oos_returns[asset]
                        if abs(1 + prev_gross_ret) > 1e-6:
                            adjusted_prev[asset] = prev_w * (1 + prev_r) / (1 + prev_gross_ret)
                        else:
                            adjusted_prev[asset] = 0.0
                    else:
                        if abs(1 + prev_gross_ret) > 1e-6:
                            adjusted_prev[asset] = prev_w / (1 + prev_gross_ret)
                        else:
                            adjusted_prev[asset] = 0.0
                
                all_assets = set(adjusted_prev.keys()) | set(common_weights.keys())
                turnover = sum(abs(common_weights.get(a, 0.0) - adjusted_prev.get(a, 0.0)) for a in all_assets)
                tc = transaction_cost * (1 + gross_return) * turnover
            else:
                turnover = sum(abs(w) for w in common_weights.values())
                tc = transaction_cost * (1 + gross_return) * turnover
            
            net_return = gross_return - tc
            
            results_storage[ptype]['returns'].append(net_return)
            results_storage[ptype]['dates'].append(current_date)
            results_storage[ptype]['weights_list'].append(common_weights.copy())
            results_storage[ptype]['turnover_list'].append(turnover)
            results_storage[ptype]['gross_returns'].append(gross_return)
            
            results_storage[ptype]['prev_weights_dict'] = common_weights.copy()
            results_storage[ptype]['prev_oos_returns_dict'] = {a: oos_returns_dict[a] for a in common_assets}
            results_storage[ptype]['prev_gross_return'] = gross_return
        
        if verbose:
            for ptype in portfolio_types:
                if len(results_storage[ptype]['returns']) > 0:
                    last_idx = len(results_storage[ptype]['returns']) - 1
                    gross_ret = results_storage[ptype]['gross_returns'][last_idx]
                    net_ret = results_storage[ptype]['returns'][last_idx]
                    to = results_storage[ptype]['turnover_list'][last_idx]
                    tc = gross_ret - net_ret
                    print(f"  {ptype.upper()}: Gross={gross_ret:>7.4f} | TO={to:>5.3f} | TC={tc:>7.5f} | Net={net_ret:>7.4f}")
    
    if verbose:
        print("\n" + "="*70)
        print("BACKTEST COMPLETE")
        print("="*70)
    
    results_dict = {}
    
    for ptype in portfolio_types:
        portfolio_returns = results_storage[ptype]['returns']
        
        if len(portfolio_returns) == 0:
            results_dict[ptype] = {'results_df': pd.DataFrame(), 'metrics': {}}
            continue
        
        results_df = pd.DataFrame({
            'date': results_storage[ptype]['dates'],
            'portfolio_return': portfolio_returns,
            'portfolio_gross_return': results_storage[ptype]['gross_returns'],
            'portfolio_turnover': results_storage[ptype]['turnover_list']
        })
        results_df['cumulative_return'] = (1 + results_df['portfolio_return']).cumprod() - 1
        
        mean_return = np.mean(portfolio_returns)
        variance = np.var(portfolio_returns, ddof=1)
        sharpe_ratio = mean_return / np.sqrt(variance) if variance > 0 else 0
        
        # Annualized metrics using 252 for daily trading
        annual_return = mean_return * 252
        annual_volatility = np.sqrt(variance * 252)
        annual_sharpe = annual_return / annual_volatility if annual_volatility > 0 else 0
        
        overall_metrics = {
            'mean_return': mean_return,
            'variance': variance,
            'sharpe_ratio': sharpe_ratio,
            'annual_return': annual_return,
            'annual_volatility': annual_volatility,
            'annual_sharpe_ratio': annual_sharpe,
            'total_return': results_df['cumulative_return'].iloc[-1],
            'avg_turnover': np.mean(results_storage[ptype]['turnover_list']),
            'n_periods': len(portfolio_returns)
        }
        
        results_dict[ptype] = {'results_df': results_df, 'metrics': overall_metrics}
    
    return results_dict

In [4]:
df = pd.read_csv('../green cleaned.csv', dtype={'ncusip': 'string'})
df['ret_fwd_1'] = (df.groupby('permno')['ret_excess'].shift(-1) )

In [5]:
df_daily = pd.read_csv('../Paper revisions/filtered_daily_data_w_ticker.csv')
df_daily['ret_fwd_1'] = df_daily.groupby('permno')['ret_excess'].shift(-1)
df_daily = df_daily.rename(columns={'date': 'datadate'})

In [6]:
variables = ['agr', 'bm', 'mom12m', 'mve', 'operprof', 'roeq', 'absacc',
            'acc', 'aeavol', 'age', 'baspread', 'BETA', 'bm_ia', 'cash',
            'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chatoia', 'chcsho',
            'chempia', 'chfeps', 'chinv', 'chmom', 'chnanalyst', 'chpmia',
            'chtx', 'cinvest', 'convind', 'currat', 'depr', 'disp', 'divi',
            'divo', 'dy', 'ear', 'egr', 'ep', 'fgr5yr', 'gma', 'grcapx',
            'grltnoa', 'herf', 'hire', 'idiovol', 'ill', 'indmom', 'invest',
            'IPO', 'lev', 'mom1m', 'mom36m', 'ms', 'mve_ia', 'nanalyst',
            'nincr', 'orgcap', 'pchcapx_ia', 'pchcurrat', 'pchdepr',
            'pchgm_pchsale', 'pchsale_pchinvt', 'pchsale_pchrect',
            'pchsale_pchxsga', 'pchsaleinv', 'pctacc', 'pricedelay', 'ps',
            'rd', 'rd_mve', 'rd_sale', 'realestate', 'retvol', 'roaq',
            'roavol', 'roic', 'rsup', 'salecash', 'saleinv', 'salerec',
            'secured', 'securedind', 'sfe', 'sgr', 'sin', 'sp', 'std_dolvol',
            'std_turn', 'stdcf', 'sue', 'tang', 'tb', 'turn', 'zerotrade']

In [7]:
results = integrated_backtest(
    df_daily = df_daily,
    df_monthly = df,
    factors_path='F-F_Research_Data_Factors_daily.csv',
    ic='GIC',  # or 'BIC', 'WIC', 'AIC', 'cv'
    test_start_date='2024-02-01',
    test_end_date='2024-04-30',
    rf_train_years=15,
    features=['mom12m', 'mve', 'bm'],
    top_q=0.90,
    bottom_q=0.10,
    lookback_window=180,
    transaction_cost=0.001,
    verbose=True
)

# Access results for each portfolio
gmv_results = results['gmv']['results_df']
gmv_metrics = results['gmv']['metrics']

mv_results = results['mv']['results_df']
mv_metrics = results['mv']['metrics']

msr_results = results['msr']['results_df']
msr_metrics = results['msr']['metrics']

# Compare Sharpe ratios
print(f"GMV Sharpe: {gmv_metrics['annual_sharpe_ratio']:.4f}")
print(f"MV Sharpe:  {mv_metrics['annual_sharpe_ratio']:.4f}")
print(f"MSR Sharpe: {msr_metrics['annual_sharpe_ratio']:.4f}")

DAILY INTEGRATED BACKTEST: RF SELECTION + RESIDUAL NODEWISE
Test Period: 2024-02-01 to 2024-04-30

TRAINING RF FOR TRADING MONTH: February 2024
Using monthly features at: 2024-01-31
  Training samples: 89406
  Training R-squared: 0.0089
  Stocks evaluated: 500
  Upper expected return bound (Top 10%): 0.0142
  Lower expected return bound (Bottom 10%): 0.0112
  Stocks selected: 100

[2024-02-01] | Assets: 36/100 valid
  Running Residual Nodewise...
  GMV: Gross=-0.0027 | TO=1.004 | TC=0.00100 | Net=-0.0037
  MV: Gross= 0.0047 | TO=2.857 | TC=0.00287 | Net= 0.0018
  MSR: Gross= 0.0009 | TO=1.603 | TC=0.00160 | Net=-0.0007

[2024-02-02] | Assets: 36/100 valid
  Running Residual Nodewise...
  GMV: Gross=-0.0040 | TO=0.014 | TC=0.00001 | Net=-0.0040
  MV: Gross=-0.0027 | TO=0.218 | TC=0.00022 | Net=-0.0029
  MSR: Gross=-0.0034 | TO=0.112 | TC=0.00011 | Net=-0.0035

[2024-02-05] | Assets: 36/100 valid
  Running Residual Nodewise...
  GMV: Gross= 0.0018 | TO=0.027 | TC=0.00003 | Net= 0.0017
  

In [9]:
print(f"\n GMV")
print(f"Annualized Sharpe Ratio: {gmv_metrics['annual_sharpe_ratio']:.4f}")
print(f"Mean Return: {gmv_metrics['mean_return']*252:.4f}")
print(f"Variance: {gmv_metrics['variance']*252:.4f}")
print(f"Avg Turnover: {gmv_metrics['avg_turnover']:.4f}")

print(f"\n MV")
print(f"Annualized Sharpe Ratio: {mv_metrics['annual_sharpe_ratio']:.4f}")
print(f"Mean Return: {mv_metrics['mean_return']*252:.4f}")
print(f"Variance: {mv_metrics['variance']*252:.4f}")
print(f"Avg Turnover: {mv_metrics['avg_turnover']:.4f}")

print(f"\n MSR")
print(f"Annualized Sharpe Ratio: {msr_metrics['annual_sharpe_ratio']:.4f}")
print(f"Mean Return: {msr_metrics['mean_return']*252:.4f}")
print(f"Variance: {msr_metrics['variance']*252:.4f}")
print(f"Avg Turnover: {msr_metrics['avg_turnover']:.4f}")


 GMV
Annualized Sharpe Ratio: 1.0046
Mean Return: 0.1025
Variance: 0.0104
Avg Turnover: 0.1063

 MV
Annualized Sharpe Ratio: 1.6570
Mean Return: 0.3907
Variance: 0.0556
Avg Turnover: 0.5521

 MSR
Annualized Sharpe Ratio: 0.5895
Mean Return: 0.1023
Variance: 0.0301
Avg Turnover: 0.3243
